# Quantum Noise Mitigation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThisIsBarney/quantx/blob/master/notebooks/09_noise_mitigation.ipynb)  [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ThisIsBarney/quantx/master?filepath=notebooks/09_noise_mitigation.ipynb)

This notebook demonstrates classical post-processing techniques to reduce noise in quantum computations:

1. **Zero-Noise Extrapolation (ZNE)** — extrapolate to zero noise by running at multiple noise levels
2. **Richardson Extrapolation** — polynomial extrapolation for smooth convergence
3. **Probabilistic Error Cancellation (PEC)** — quasi-probability decomposition
4. **Readout Error Mitigation** — calibration matrix inversion

All examples run in pure Python (no C++ library required).

In [ ]:
# Install quantx-engine (needed in Colab / Binder)
try:
    import quantx_engine  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'quantx-engine'])
    import quantx_engine  # noqa: F401
print('quantx_engine version:', quantx_engine.__version__)

In [ ]:
import numpy as np
import quantx_engine
print('quantx_engine version:', quantx_engine.__version__)

## 1. Zero-Noise Extrapolation (ZNE)

ZNE amplifies the noise by scale factors $\lambda_1 < \lambda_2 < \ldots$ (e.g., gate folding),
measures expectation values $E(\lambda_i)$, then fits a model and extrapolates to $\lambda=0$.

### Setting up an executor

We build a simple noisy statevector executor using `make_sv_executor` and depolarizing noise.

In [ ]:
from quantx_engine.mitigation import ZNEMitigator, zne_mitigate, make_sv_executor
from quantx_engine.noise import DepolarizingNoise, NoiseModel
from quantx_engine.circuit import Circuit

# Build a simple 2-qubit Bell-state circuit and measure ⟨Z⊗Z⟩
# For an ideal Bell state |Φ+⟩ = (|00⟩+|11⟩)/√2, ⟨Z⊗Z⟩ = 1

bell_circuit = {
    'n_qubits': 2,
    'gates': [
        {'gate': 'H',  'qubits': [0]},
        {'gate': 'CX', 'qubits': [0, 1]},
    ]
}

# Observable: Z⊗Z (expectation value 1 for Bell state)
observable = {'label': 'Z0Z1', 'coefficient': 1.0}

# Noise level: 1% depolarizing on each gate
p_noise = 0.02
noise_model = NoiseModel()
noise_model.add(DepolarizingNoise(prob=p_noise))

executor = make_sv_executor(bell_circuit, observable, noise_model=noise_model)

# Without mitigation
noisy_val = executor(scale_factor=1.0)
print(f'Noisy ⟨Z⊗Z⟩ at scale 1.0: {noisy_val:.6f}  (ideal: 1.0)')

In [ ]:
# ZNE with polynomial extrapolation
scale_factors = [1.0, 2.0, 3.0]

result = zne_mitigate(
    executor=executor,
    scale_factors=scale_factors,
    extrapolator='poly',   # polynomial fit
)

print('ZNE results (polynomial extrapolation):')
print(f'  Scale factors: {scale_factors}')
print(f'  Raw values:    {[f"{v:.4f}" for v in result.raw_values]}')
print(f'  Mitigated ⟨Z⊗Z⟩: {result.mitigated_value:.6f}  (ideal: 1.0)')
print(f'  Error reduction: {abs(noisy_val - 1.0):.4f} → {abs(result.mitigated_value - 1.0):.4f}')

In [ ]:
# ZNE with exponential extrapolation
result_exp = zne_mitigate(
    executor=executor,
    scale_factors=scale_factors,
    extrapolator='exp',
)

print('ZNE with exponential extrapolation:')
print(f'  Mitigated ⟨Z⊗Z⟩: {result_exp.mitigated_value:.6f}  (ideal: 1.0)')

# ZNE with Richardson extrapolation
result_rich = zne_mitigate(
    executor=executor,
    scale_factors=scale_factors,
    extrapolator='richardson',
)

print(f'ZNE with Richardson extrapolation:')
print(f'  Mitigated ⟨Z⊗Z⟩: {result_rich.mitigated_value:.6f}  (ideal: 1.0)')

print()
print(f'Summary (ideal = 1.0, noisy = {noisy_val:.4f}):')
print(f'  Polynomial:   {result.mitigated_value:.6f}')
print(f'  Exponential:  {result_exp.mitigated_value:.6f}')
print(f'  Richardson:   {result_rich.mitigated_value:.6f}')

## 2. Richardson Extrapolation

`RichardsonExtrapolation` fits the noise-scaled expectation values
$E(\lambda_i)$ with a polynomial and extrapolates to $\lambda \to 0$.

It is equivalent to the Richardson polynomial limit and achieves $(n-1)$-th order
noise cancellation with $n$ scale factors.

In [ ]:
from quantx_engine.mitigation import RichardsonExtrapolation

# Synthetic data: E(λ) = 1.0 - 0.1*λ - 0.02*λ² (model a noisy expectation value)
def true_func(lam):
    return 1.0 - 0.1 * lam - 0.02 * lam**2

scale_factors = [1.0, 2.0, 3.0, 4.0]
values = [true_func(s) for s in scale_factors]

extrapolator = RichardsonExtrapolation(scale_factors, values)
mitigated = extrapolator.extrapolate()

true_zero = true_func(0.0)  # = 1.0

print('Richardson Extrapolation example:')
print(f'  Scale factors: {scale_factors}')
print(f'  Values:        {[f"{v:.4f}" for v in values]}')
print(f'  Extrapolated:  {mitigated:.8f}')
print(f'  True zero:     {true_zero:.8f}')
print(f'  Error:         {abs(mitigated - true_zero):.2e}')

assert abs(mitigated - true_zero) < 1e-8, f'Richardson extrapolation error too large: {abs(mitigated - true_zero)}'
print('Accuracy check passed  ✓')

In [ ]:
# Convergence: more scale factors → better accuracy for polynomial noise models
print('Richardson convergence with increasing number of scale factors:')
print(f'{"n_scales":>10}  {"extrapolated":>14}  {"error":>12}')
print('-' * 42)

for n in [2, 3, 4, 5]:
    sf = list(range(1, n + 1))
    vals = [true_func(s) for s in sf]
    ext = RichardsonExtrapolation(sf, vals).extrapolate()
    err = abs(ext - 1.0)
    print(f'{n:>10}  {ext:>14.8f}  {err:>12.2e}')

## 3. Probabilistic Error Cancellation (PEC)

PEC represents the ideal gate as a quasi-probability distribution over implementable noisy operations.
We use the `QuasiProbabilityDecomposition` for a depolarizing noise model.

In [ ]:
from quantx_engine.mitigation import QuasiProbabilityDecomposition, pec_mitigate

# Depolarizing channel with p=0.02
p = 0.02
qpd = QuasiProbabilityDecomposition.from_depolarizing(p, n_qubits=1)

print('Quasi-probability decomposition for depolarizing(p=0.02):')
print(f'  1-norm (sampling overhead): {qpd.one_norm:.4f}')
print(f'  Number of Pauli channels:   {len(qpd.terms)}')
for label, coeff in sorted(qpd.terms.items()):
    print(f'    {label}: {coeff:+.4f}')

In [ ]:
# PEC mitigation of the Bell circuit
pec_result = pec_mitigate(
    executor=executor,
    qpd=qpd,
    n_samples=200,
    seed=42,
)

print(f'PEC mitigation of Bell ⟨Z⊗Z⟩:')
print(f'  Noisy value:    {noisy_val:.6f}')
print(f'  PEC mitigated:  {pec_result.mitigated_value:.6f}  (ideal: 1.0)')
print(f'  Sampling overhead (1-norm²): {qpd.one_norm**2:.2f}x')

# PEC is unbiased — mitigated value should be closer to ideal
print(f'  Error: {abs(pec_result.mitigated_value - 1.0):.4f}')

## 4. Noise Model Library

QuantX includes a comprehensive noise model library. Here we demonstrate
building composite noise models and inspecting their Kraus operators.

In [ ]:
from quantx_engine.noise import (
    DepolarizingNoise,
    AmplitudeDampingNoise,
    ThermalRelaxationNoise,
    ReadoutNoise,
    NoiseModel,
)

# Build a realistic composite noise model
model = NoiseModel()
model.add(DepolarizingNoise(prob=0.005))          # 0.5% depolarizing
model.add(ThermalRelaxationNoise(
    T1=100e-6,   # 100 µs T1 relaxation time
    T2=60e-6,    # 60 µs T2 dephasing time
    t_gate=50e-9 # 50 ns gate time
))
model.add(ReadoutNoise(p01=0.02, p10=0.03))       # readout errors

model_dict = model.to_dict()

print('Composite noise model:')
print(f'  Channels: {len(model_dict["channels"])}')
for ch in model_dict['channels']:
    print(f'  - {ch["type"]}: {ch.get("params", {})}')

In [ ]:
# Inspect Kraus operators for amplitude damping
from quantx_engine.noise import AmplitudeDampingNoise

gamma = 0.1  # 10% decay probability
adc = AmplitudeDampingNoise(gamma=gamma)
kraus_ops = adc.kraus_operators()

print(f'Amplitude Damping (γ={gamma}) Kraus operators:')
for i, K in enumerate(kraus_ops):
    print(f'  K_{i} =\n{K}')

# Verify completeness: Σ K†K = I
completeness = sum(K.conj().T @ K for K in kraus_ops)
print(f'\nCompleteness Σ K†K = I:')
print(completeness)
assert np.allclose(completeness, np.eye(2), atol=1e-10), 'Kraus completeness failed!'
print('Completeness check passed  ✓')

## 5. Noise Characterization via Randomized Benchmarking

The `StandardRB` class runs a randomized benchmarking experiment to estimate
the average gate error per Clifford (EPC).

In [ ]:
from quantx_engine.rb_qv import StandardRB

# Simulated depolarizing noise at 1% per gate
p_gate = 0.01
noise = NoiseModel()
noise.add(DepolarizingNoise(prob=p_gate))

# Run RB with sequences of lengths [1, 2, 4, 8, 16, 32]
rb = StandardRB(
    n_qubits=1,
    sequence_lengths=[1, 2, 4, 8, 16, 32],
    n_sequences=20,
    noise_model=noise,
    seed=42,
)
rb_result = rb.run()

# EPC from fit
epc = rb_result.epc
print(f'Randomized Benchmarking:')
print(f'  Injected gate error: {p_gate:.4f}')
print(f'  Fitted EPC:          {epc:.4f}')
print(f'  A (amplitude):       {rb_result.fit_params["A"]:.4f}')
print(f'  p (decay rate):      {rb_result.fit_params["p"]:.4f}')

# EPC should be close to the injected error
assert abs(epc - p_gate) < 0.05, f'RB EPC {epc:.4f} far from injected {p_gate:.4f}'
print('EPC consistency check passed  ✓')

## Summary

| Technique | Key benefit | QuantX module |
|-----------|------------|---------------|
| ZNE (polynomial) | Simple, no model of noise needed | `mitigation.zne_mitigate` |
| ZNE (Richardson) | Exact for polynomial noise, order-$(n-1)$ cancellation | `mitigation.RichardsonExtrapolation` |
| PEC | Unbiased, treats arbitrary noise | `mitigation.pec_mitigate` |
| Noise model library | Build composite noise for simulation | `noise.NoiseModel` |
| Randomized benchmarking | Measure average gate fidelity in hardware | `rb_qv.StandardRB` |

**When to use which technique:**
- **ZNE**: Low overhead, great for near-term devices; polynomial extrapolation works well for smooth noise
- **Richardson**: Best when noise scales polynomially with gate count
- **PEC**: Optimal for known noise models; overhead grows as $(1+2p)^{2n}$
- **RB**: Characterize your device before choosing a mitigation strategy

Next: **[01 — Getting Started](01_getting_started.ipynb)**